<a href="https://colab.research.google.com/github/vernqwen/easyshare/blob/main/YTDLP_WEB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import sys
import subprocess
import shutil
import datetime
from pathlib import Path

# --- SETUP DIREKTORI GOOGLE COLAB ---
BASE_DIR = Path("/content")
DOWNLOADS_DIR = BASE_DIR / "downloads"
TEMP_DIR = BASE_DIR / "temp"
CONFIG_FILE = BASE_DIR / "yt-dlp.conf"
COOKIES_FILE = BASE_DIR / "cookies.txt"
HISTORY_LOG = BASE_DIR / "download_history.log"
VERBOSE_LOG = BASE_DIR / "atomic_debug.log"

DOWNLOADS_DIR.mkdir(parents=True, exist_ok=True)
TEMP_DIR.mkdir(parents=True, exist_ok=True)

# Map dependency Linux (tanpa akhiran .exe)
CORE_DEPENDENCIES = {
    "yt-dlp": "Core Extractor & Crawler Engine",
    "aria2c": "Multi-stream Connection Accelerator",
    "ffmpeg": "A/V Muxer & Transcoder",
    "ffprobe": "Media Stream & Codec Inspector",
    "streamlink": "Modern Live Streaming Extractor",
    "node": "JavaScript Runtime (YouTube Cipher/n-token)",
    "curl": "Advanced Network Transfer Utility",
    "AtomicParsley": "MP4/M4A Atom Metadata Tagger",
    "MP4Box": "GPAC ISO MP4 Multiplexer",
    "mkvmerge": "Matroska (MKV) Multiplexer",
    "mkvextract": "MKV Track/Subtitle Extractor",
    "mkvpropedit": "MKV Header & Track Property Editor",
}

# Global Advanced Settings State
GLOBAL_SETTINGS = {
    "use_aria2c": True,
    "concurrent_fragments": "16",
    "rate_limit": None,
    "socket_timeout": "30",
    "playlist_mode": "yes",  # 'yes' atau 'no'
}

def install_environment_dependencies():
    """Fungsi otomatisasi penginstalan semua perkakas di Google Colab"""
    print("[*] Mengunduh & Menginstal Dependencies Sistem di Linux Colab...")

    # Update dan Install Apt Packages
    apt_packages = [
        "ffmpeg", "aria2", "streamlink", "curl",
        "atomicparsley", "gpac", "mkvtoolnix", "nodejs"
    ]
    subprocess.run(["apt-get", "update", "-qq"], stdout=subprocess.DEVNULL)
    subprocess.run(["apt-get", "install", "-y", "-qq"] + apt_packages, stdout=subprocess.DEVNULL)

    # Update/Install yt-dlp via Pip
    subprocess.run([sys.executable, "-m", "pip", "install", "-U", "yt-dlp", "streamlink"], stdout=subprocess.DEVNULL)
    print("[+] Instalasi Environment Selesai!\n")

def write_history(url, status, detail=""):
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    with open(HISTORY_LOG, "a", encoding="utf-8") as f:
        f.write(f"[{timestamp}] STATUS: {status} | URL: {url} | DETAIL: {detail}\n")

def check_binaries():
    print("=" * 80)
    print("      STATUS VERIFIKASI BINARY DEPENDENCIES (LINUX ENVIRONMENT)      ")
    print("=" * 80)
    all_ok = True
    for bin_name, desc in CORE_DEPENDENCIES.items():
        is_ready = shutil.which(bin_name) is not None
        status = "[ OK ]" if is_ready else "[TIDAK ADA]"
        if not is_ready:
            all_ok = False
        print(f" {status:<12} | {bin_name:<18} | {desc}")
    print("=" * 80)
    return all_ok

def get_current_ytdlp_channel():
    try:
        output = subprocess.check_output(["yt-dlp", "--version"], text=True, stderr=subprocess.STDOUT)
        if "nightly" in output.lower():
            return "nightly"
        elif "master" in output.lower():
            return "master"
        return "stable"
    except Exception:
        return "unknown"

def manage_ytdlp_version():
    while True:
        current_channel = get_current_ytdlp_channel()
        print("\n--- MANAJEMEN VERSI DAN CHANNEL YT-DLP ---")
        print("1. Cek Versi yt-dlp Saat Ini")
        print(f"2. Perbarui yt-dlp ke Versi Terbaru (Channel Aktif: {current_channel.upper()})")
        print("3. Ubah Channel ke Nightly")
        print("4. Ubah Channel ke Master")
        print("5. Kembali ke Menu Utama")
        pilihan = input("Pilih menu (1-5): ").strip()

        if pilihan == "1":
            subprocess.run(["yt-dlp", "--version"])
        elif pilihan == "2":
            subprocess.run(["yt-dlp", "-U"])
        elif pilihan == "3":
            subprocess.run(["yt-dlp", "--update-to", "nightly"])
        elif pilihan == "4":
            subprocess.run(["yt-dlp", "--update-to", "master"])
        elif pilihan == "5":
            break

def configure_advanced_options():
    print("\n--- PENGATURAN ADVANCED UNDUHAN ---")
    print(f"1. Toggle Aria2c Engine (Saat ini: {'AKTIF' if GLOBAL_SETTINGS['use_aria2c'] else 'NON-AKTIF'})")
    print(f"2. Set Fragment Unduhan Bersamaan (1-25) (Saat ini: {GLOBAL_SETTINGS['concurrent_fragments']})")
    print(f"3. Set Batas Laju / Speed Limit (cth: 50K, 2M) (Saat ini: {GLOBAL_SETTINGS['rate_limit'] or 'Tanpa Batas'})")
    print(f"4. Set Socket Timeout dalam detik (Saat ini: {GLOBAL_SETTINGS['socket_timeout']}s)")
    print(f"5. Mode Playlist (Saat ini: {'Unduh Semua Playlist' if GLOBAL_SETTINGS['playlist_mode']=='yes' else 'Unduh Single Video Saja'})")
    print("6. Selesai / Kembali")

    pilih = input("Pilihan (1-6): ").strip()
    if pilih == "1":
        GLOBAL_SETTINGS["use_aria2c"] = not GLOBAL_SETTINGS["use_aria2c"]
    elif pilih == "2":
        val = input("Masukkan jumlah fragmen (1-25): ").strip()
        if val.isdigit() and 1 <= int(val) <= 25:
            GLOBAL_SETTINGS["concurrent_fragments"] = val
    elif pilih == "3":
        val = input("Masukkan limit speed (kosongkan untuk unlimited, cth 500K atau 2M): ").strip()
        GLOBAL_SETTINGS["rate_limit"] = val if val else None
    elif pilih == "4":
        val = input("Masukkan timeout (detik): ").strip()
        if val.isdigit():
            GLOBAL_SETTINGS["socket_timeout"] = val
    elif pilih == "5":
        GLOBAL_SETTINGS["playlist_mode"] = "no" if GLOBAL_SETTINGS["playlist_mode"] == "yes" else "yes"

def build_base_cmd():
    cmd = ["yt-dlp"]
    if CONFIG_FILE.exists():
        cmd.extend(["--config-locations", str(CONFIG_FILE)])

    cmd.extend(["-P", f"temp:{TEMP_DIR}"])
    cmd.extend(["-P", f"home:{DOWNLOADS_DIR}"])
    cmd.extend(["-o", "%(title)s [%(id)s].%(ext)s"])

    cmd.append("--no-keep-fragments")
    cmd.extend(["--parse-metadata", "%(uploader)s:%(meta_artist)s"])

    if GLOBAL_SETTINGS["use_aria2c"]:
        cmd.extend(["--downloader", "aria2c", "--downloader-args", f"aria2c:-x {GLOBAL_SETTINGS['concurrent_fragments']} -s {GLOBAL_SETTINGS['concurrent_fragments']}"])
    else:
        cmd.extend(["--concurrent-fragments", GLOBAL_SETTINGS["concurrent_fragments"]])

    if GLOBAL_SETTINGS["rate_limit"]:
        cmd.extend(["--limit-rate", GLOBAL_SETTINGS["rate_limit"]])

    cmd.extend(["--socket-timeout", GLOBAL_SETTINGS["socket_timeout"]])

    if GLOBAL_SETTINGS["playlist_mode"] == "no":
        cmd.append("--no-playlist")
    else:
        cmd.append("--yes-playlist")

    return cmd

def menu_resolusi_solid():
    print("\n" + "=" * 50)
    print("A. PILIH RESOLUSI UNDUHAN SOLID")
    print("=" * 50)
    print("AA. VIDEO + AUDIO")
    print("  1. Kualitas Terbaik Otomatis")
    print("  2. Kualitas 1440p / 2160p (4K) / 4320p (8K) Asli Video")
    print("  3. Kualitas 1080p Kunci Asli")
    print("  4. Kualitas 720p Kunci Asli")
    print("  5. Kualitas Rendah (140p / 144p / 240p / 360p / 480p)")
    print("BA. AUDIO SAJA")
    print("  6. Audio Terbaik (320kbps)")
    print("  7. High Audio")
    print("  8. Medium Audio")
    print("  9. Low Audio")
    print("CA. VIDEO SAJA (TANPA AUDIO)")
    print("  10. Video Saja Kualitas Terbaik")
    print("  11. Video Saja 1440p / 2160p / 4320p Asli")
    print("  12. Video Saja 1080p Kunci Asli")
    print("  13. Video Saja 720p Kunci Asli")
    print("  14. Video Saja Rendah (144p - 480p)")

    sub_pilihan = input("\nMasukkan Nomor Pilihan Resolusi (1-14): ").strip()

    fmt = "bestvideo+bestaudio/best"
    if sub_pilihan == "2":
        fmt = "bestvideo[height>=1440]+bestaudio/best"
    elif sub_pilihan == "3":
        fmt = "bestvideo[height<=1080][height>=1080]+bestaudio/best[height<=1080]"
    elif sub_pilihan == "4":
        fmt = "bestvideo[height<=720][height>=720]+bestaudio/best[height<=720]"
    elif sub_pilihan == "5":
        fmt = "bestvideo[height<=480]+bestaudio/best[height<=480]"
    elif sub_pilihan == "6":
        return ["-x", "--audio-format", "mp3", "--audio-quality", "0"]
    elif sub_pilihan == "7":
        return ["-x", "--audio-format", "m4a", "--audio-quality", "2"]
    elif sub_pilihan == "8":
        return ["-x", "--audio-format", "mp3", "--audio-quality", "5"]
    elif sub_pilihan == "9":
        return ["-x", "--audio-format", "mp3", "--audio-quality", "9"]
    elif sub_pilihan == "10":
        fmt = "bestvideo"
    elif sub_pilihan == "11":
        fmt = "bestvideo[height>=1440]"
    elif sub_pilihan == "12":
        fmt = "bestvideo[height<=1080][height>=1080]"
    elif sub_pilihan == "13":
        fmt = "bestvideo[height<=720][height>=720]"
    elif sub_pilihan == "14":
        fmt = "bestvideo[height<=480]"

    return ["-f", fmt]

def process_single_url(target_url, extra_args):
    cmd = build_base_cmd() + extra_args

    print("\n--- FITUR PROSESING TAMBAHAN ---")
    thumb = input("Sertakan Thumbnail Cover? (y/n, default=y): ").strip().lower()
    if thumb == 'n':
        cmd.append("--no-embed-thumbnail")
    else:
        cmd.append("--embed-thumbnail")

    print("\nPilih Konversi Container Video:")
    print("1. Default (MKV/MP4 Auto)  2. MP4  3. MKV  4. AVI  5. MOV  6. WEBM")
    c_opt = input("Pilihan Container (1-6): ").strip()
    container_map = {"2": "mp4", "3": "mkv", "4": "avi", "5": "mov", "6": "webm"}
    if c_opt in container_map:
        cmd.extend(["--merge-output-format", container_map[c_opt], "--recode-video", container_map[c_opt]])

    sub_opt = input("\nUnduh Takarir/Subtitle? (1. Input Langsung ke Video / 2. Unduh Terpisah / 3. Tanpa Subtitle): ").strip()
    if sub_opt == "1":
        lang = input("Masukkan kode bahasa (cth: id,en atau 'all'): ").strip() or "all"
        cmd.extend(["--embed-subs", "--sub-langs", lang])
    elif sub_opt == "2":
        lang = input("Masukkan kode bahasa (cth: id,en atau 'all'): ").strip() or "all"
        cmd.extend(["--write-subs", "--sub-langs", lang, "--skip-download"])

    clipper = input("\nIngin Memotong Segmen Video tertentu? (y/n): ").strip().lower()
    if clipper == 'y':
        start_t = input("Waktu Mulai (format HH:MM:SS atau Detik): ").strip()
        end_t = input("Waktu Selesai (format HH:MM:SS atau Detik): ").strip()
        cmd.extend(["--download-sections", f"*{start_t}-{end_t}"])

    live_opt = input("\nApakah Ini Video Siaran Langsung / Live Stream? (y/n): ").strip().lower()
    if live_opt == 'y':
        cmd.extend(["--from-title", ".*", "--live-from-start"])

    cmd.append(target_url)

    print("\n[*] Menjalankan pipeline unduhan untuk URL:", target_url)
    try:
        subprocess.run(cmd, check=True)
        write_history(target_url, "BERHASIL", f"Format: {extra_args}")
    except Exception as e:
        print(f"\n[!] Error eksekusi: {e}")
        write_history(target_url, "GAGAL", str(e))

def handle_atomic_logging():
    print("\n[*] Mengekstraksi Log Universal & Status Atomic Program...")
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    with open(VERBOSE_LOG, "w", encoding="utf-8") as f:
        f.write("=" * 80 + "\n")
        f.write(f"      SYSTEM DIAGNOSTIC & ATOMIC STATE LOG - {timestamp}\n")
        f.write("=" * 80 + "\n\n")

        f.write("[1] PYTHON & ENVIRONMENT PATH STATE\n")
        f.write(f"Python Executable : {sys.executable}\n")
        f.write(f"Python Version    : {sys.version}\n")
        f.write(f"Base Directory    : {BASE_DIR}\n")
        f.write(f"System PATH       : {os.environ.get('PATH')}\n\n")

        f.write("[2] BINARY DEPENDENCIES STATUS\n")
        for bin_name, desc in CORE_DEPENDENCIES.items():
            bin_path = shutil.which(bin_name)
            exists = bin_path is not None
            f.write(f"Binary: {bin_name:<20} | Status: {'OK' if exists else 'MISSING':<7} | Path: {bin_path}\n")
        f.write("\n")

        f.write("[3] RUNTIME EXECUTABLE VERSION DIAGNOSTICS\n")
        try:
            ytdlp_ver = subprocess.check_output(["yt-dlp", "--version"], text=True, stderr=subprocess.STDOUT).strip()
            f.write(f"yt-dlp Version    : {ytdlp_ver}\n")
        except Exception as e:
            f.write(f"yt-dlp Version    : ERROR ({e})\n")

        try:
            ffmpeg_ver = subprocess.check_output(["ffmpeg", "-version"], text=True, stderr=subprocess.STDOUT).splitlines()[0]
            f.write(f"FFmpeg Version    : {ffmpeg_ver}\n")
        except Exception as e:
            f.write(f"FFmpeg Version    : ERROR ({e})\n")
        f.write("\n")

        f.write("[4] CURRENT GLOBAL SETTINGS STATE\n")
        for k, v in GLOBAL_SETTINGS.items():
            f.write(f"Setting {k:<22} : {v}\n")
        f.write("\n")

        f.write("[5] CONFIGURATION FILE CONTENT (yt-dlp.conf)\n")
        if CONFIG_FILE.exists():
            f.write(CONFIG_FILE.read_text(encoding="utf-8") + "\n\n")
        else:
            f.write("FILE yt-dlp.conf TIDAK DITEMUKAN!\n\n")

        f.write("[6] RECENT DOWNLOAD HISTORY LOG\n")
        if HISTORY_LOG.exists():
            lines = HISTORY_LOG.read_text(encoding="utf-8").splitlines()
            f.write("\n".join(lines[-30:]) + "\n")
        else:
            f.write("Belum ada riwayat unduhan.\n")

        f.write("\n" + "=" * 80 + "\n")
        f.write("END OF DIAGNOSTIC LOG REPORT\n")
        f.write("=" * 80 + "\n")

    print(f"[+] Log status universal berhasil diekstrak penuh ke: {VERBOSE_LOG}")

def main():
    install_environment_dependencies()
    check_binaries()

    while True:
        print("\n" + "=" * 80)
        print("                  ULTIMATE VIDEO DOWNLOADER SUITE - MAIN MENU                  ")
        print("=" * 80)
        print("1. Unduh Single URL / Multi-Link Sekaligus (Proses Pipeline Utama)")
        print("2. Cek Verifikasi Dependencies Binary Linux")
        print("3. Manajemen Versi & Channel yt-dlp (Nightly/Stable/Master)")
        print("4. Pengaturan Mode Advanced (Aria2c, Limit Speed, Timeout, Fragmen, Playlist)")
        print("5. Ekstraksi File Log Detail Rinci (Atom Level Debug Logging)")
        print("6. Keluar dari Aplikasi")

        main_choice = input("\nPilih Menu Utama (1-6): ").strip()

        if main_choice == "2":
            check_binaries()
        elif main_choice == "3":
            manage_ytdlp_version()
        elif main_choice == "4":
            configure_advanced_options()
        elif main_choice == "5":
            handle_atomic_logging()
        elif main_choice == "6":
            print("\nTerima kasih telah menggunakan Ultimate Video Downloader Suite!")
            break
        elif main_choice == "1":
            print("\n--- INPUT TARGET URL ---")
            print("Anda bisa menempel satu URL atau banyak URL sekaligus (pisahkan dengan koma atau spasi).")
            raw_input_url = input("Masukkan URL Target: ").strip()
            if not raw_input_url:
                print("[!] URL tidak boleh kosong.")
                continue

            urls = [u.strip() for u in raw_input_url.replace(" ", ",").split(",") if u.strip()]

            print("\n--- METODE PEMILIHAN FORMAT ---")
            print("A. Pilih Resolusi Unduhan Solid (Daftar Preset Praktis)")
            print("B. Pilihan Resolusi Hasil Ekstrak Dinamis (-F Engine Inspector)")
            mode_fmt = input("Pilih Metode (A/B): ").strip().upper()

            extra_args = []
            if mode_fmt == "B":
                print("\n[*] Menjalankan Ekstraksi Engine yt-dlp -F terhadap URL pertama...\n")
                try:
                    res = subprocess.run(["yt-dlp", "-F", urls[0]], capture_output=True, text=True, check=True)
                    print(res.stdout)
                except subprocess.CalledProcessError as e:
                    print(f"[!] Gagal mengekstraksi format: {e}")
                    if e.stderr:
                        print(e.stderr)

                selected_id = input("\nMasukkan Format ID Pilihan Anda (contoh: 16+2 atau cukup masukkan 16): ").strip()
                if selected_id:
                    if selected_id.isdigit():
                        fmt_str = f"{selected_id}+bestaudio/best"
                    else:
                        fmt_str = selected_id
                else:
                    fmt_str = "bestvideo+bestaudio/best"
                extra_args = ["-f", fmt_str]
            else:
                extra_args = menu_resolusi_solid()

            print(f"\n[+] Total {len(urls)} URL terdeteksi.")
            exec_mode = "1"
            if len(urls) > 1:
                print("1. Proses Unduh Satu per Satu secara Berurutan")
                print("2. Proses Unduh Sekaligus (Batch URL Download)")
                exec_mode = input("Pilihan Eksekusi Batch (1/2): ").strip()

            if exec_mode == "2":
                for target in urls:
                    process_single_url(target, extra_args)
            else:
                for target in urls:
                    process_single_url(target, extra_args)

if __name__ == "__main__":
    main()

[*] Mengunduh & Menginstal Dependencies Sistem di Linux Colab...
[+] Instalasi Environment Selesai!

      STATUS VERIFIKASI BINARY DEPENDENCIES (LINUX ENVIRONMENT)      
 [ OK ]       | yt-dlp             | Core Extractor & Crawler Engine
 [ OK ]       | aria2c             | Multi-stream Connection Accelerator
 [ OK ]       | ffmpeg             | A/V Muxer & Transcoder
 [ OK ]       | ffprobe            | Media Stream & Codec Inspector
 [ OK ]       | streamlink         | Modern Live Streaming Extractor
 [ OK ]       | node               | JavaScript Runtime (YouTube Cipher/n-token)
 [ OK ]       | curl               | Advanced Network Transfer Utility
 [ OK ]       | AtomicParsley      | MP4/M4A Atom Metadata Tagger
 [ OK ]       | MP4Box             | GPAC ISO MP4 Multiplexer
 [ OK ]       | mkvmerge           | Matroska (MKV) Multiplexer
 [ OK ]       | mkvextract         | MKV Track/Subtitle Extractor
 [ OK ]       | mkvpropedit        | MKV Header & Track Property Editor

       

KeyboardInterrupt: Interrupted by user

In [ ]:
import os
import sys
import subprocess
import shutil
import datetime
from pathlib import Path
from google.colab import drive
from yt_dlp import YoutubeDL

# --- MOUNT GOOGLE DRIVE ---
drive.mount('/content/drive', force_remount=True)

# --- SETUP DIREKTORI UTAMA GOOGLE DRIVE ---
DRIVE_BASE = Path("/content/drive/MyDrive/YT_Online_Downloads")
DOWNLOADS_DIR = DRIVE_BASE / "downloads"
TEMP_DIR = DRIVE_BASE / "temp"
CONFIG_FILE = DRIVE_BASE / "yt-dlp.conf"
COOKIES_FILE = DRIVE_BASE / "cookies.txt"
HISTORY_LOG = DRIVE_BASE / "download_history.log"
VERBOSE_LOG = DRIVE_BASE / "atomic_debug.log"

# Memastikan direktori penyimpanan di Google Drive siap
DOWNLOADS_DIR.mkdir(parents=True, exist_ok=True)
TEMP_DIR.mkdir(parents=True, exist_ok=True)

# Map dependency Linux untuk verifikasi
CORE_DEPENDENCIES = {
    "yt-dlp": "Core Extractor & Crawler Engine",
    "aria2c": "Multi-stream Connection Accelerator",
    "ffmpeg": "A/V Muxer & Transcoder",
    "ffprobe": "Media Stream & Codec Inspector",
    "streamlink": "Modern Live Streaming Extractor",
    "node": "JavaScript Runtime (YouTube Cipher/n-token)",
    "curl": "Advanced Network Transfer Utility",
    "AtomicParsley": "MP4/M4A Atom Metadata Tagger",
    "MP4Box": "GPAC ISO MP4 Multiplexer",
    "mkvmerge": "Matroska (MKV) Multiplexer",
    "mkvextract": "MKV Track/Subtitle Extractor",
    "mkvpropedit": "MKV Header & Track Property Editor",
}

# Global Advanced Settings State
GLOBAL_SETTINGS = {
    "use_aria2c": True,
    "concurrent_fragments": "16",
    "rate_limit": None,
    "socket_timeout": "30",
    "playlist_mode": "yes",  # 'yes' atau 'no'
}

def progress_hook(d):
    """Fungsi hook untuk memantau dan menampilkan proses unduhan secara real-time (1%-100%)."""
    if d["status"] == "downloading":
        downloaded = d.get("downloaded_bytes", 0)
        total = d.get("total_bytes") or d.get("total_bytes_estimate", 0)

        if total > 0:
            percentage = (downloaded / total) * 100
            speed = d.get("_speed_str", "N/A")
            eta = d.get("_eta_str", "N/A")

            sys.stdout.write(
                f"\r[*] Menjalankan pipeline unduhan untuk URL | Progress: {percentage:6.2f}% | Kecepatan: {speed} | Sisa Waktu: {eta}"
            )
            sys.stdout.flush()

    elif d["status"] == "finished":
        sys.stdout.write(
            "\n[+] Unduhan selesai! Melanjutkan ke proses penggabungan/konversi...\n"
        )
        sys.stdout.flush()

def write_history(url, status, detail=""):
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    with open(HISTORY_LOG, "a", encoding="utf-8") as f:
        f.write(f"[{timestamp}] STATUS: {status} | URL: {url} | DETAIL: {detail}\n")

def check_binaries():
    print("=" * 80)
    print("      STATUS VERIFIKASI BINARY DEPENDENCIES (LINUX ENVIRONMENT)      ")
    print("=" * 80)
    all_ok = True
    for bin_name, desc in CORE_DEPENDENCIES.items():
        is_ready = shutil.which(bin_name) is not None
        status = "[ OK ]" if is_ready else "[TIDAK ADA]"
        if not is_ready:
            all_ok = False
        print(f" {status:<12} | {bin_name:<18} | {desc}")
    print("=" * 80)
    return all_ok

def get_current_ytdlp_channel():
    try:
        output = subprocess.check_output(["yt-dlp", "--version"], text=True, stderr=subprocess.STDOUT)
        if "nightly" in output.lower():
            return "nightly"
        elif "master" in output.lower():
            return "master"
        return "stable"
    except Exception:
        return "unknown"

def manage_ytdlp_version():
    while True:
        current_channel = get_current_ytdlp_channel()
        print("\n--- MANAJEMEN VERSI DAN CHANNEL YT-DLP ---")
        print("1. Cek Versi yt-dlp Saat Ini")
        print(f"2. Perbarui yt-dlp ke Versi Terbaru (Channel Aktif: {current_channel.upper()})")
        print("3. Ubah Channel ke Nightly")
        print("4. Ubah Channel ke Master")
        print("5. Kembali ke Menu Utama")
        pilihan = input("Pilih menu (1-5): ").strip()

        if pilihan == "1":
            subprocess.run(["yt-dlp", "--version"])
        elif pilihan == "2":
            subprocess.run(["yt-dlp", "-U"])
        elif pilihan == "3":
            subprocess.run(["yt-dlp", "--update-to", "nightly"])
        elif pilihan == "4":
            subprocess.run(["yt-dlp", "--update-to", "master"])
        elif pilihan == "5":
            break

def configure_advanced_options():
    print("\n--- PENGATURAN ADVANCED UNDUHAN ---")
    print(f"1. Toggle Aria2c Engine (Saat ini: {'AKTIF' if GLOBAL_SETTINGS['use_aria2c'] else 'NON-AKTIF'})")
    print(f"2. Set Fragment Unduhan Bersamaan (1-25) (Saat ini: {GLOBAL_SETTINGS['concurrent_fragments']})")
    print(f"3. Set Batas Laju / Speed Limit (cth: 50K, 2M) (Saat ini: {GLOBAL_SETTINGS['rate_limit'] or 'Tanpa Batas'})")
    print(f"4. Set Socket Timeout dalam detik (Saat ini: {GLOBAL_SETTINGS['socket_timeout']}s)")
    print(f"5. Mode Playlist (Saat ini: {'Unduh Semua Playlist' if GLOBAL_SETTINGS['playlist_mode']=='yes' else 'Unduh Single Video Saja'})")
    print("6. Selesai / Kembali")

    pilih = input("Pilihan (1-6): ").strip()
    if pilih == "1":
        GLOBAL_SETTINGS["use_aria2c"] = not GLOBAL_SETTINGS["use_aria2c"]
    elif pilih == "2":
        val = input("Masukkan jumlah fragmen (1-25): ").strip()
        if val.isdigit() and 1 <= int(val) <= 25:
            GLOBAL_SETTINGS["concurrent_fragments"] = val
    elif pilih == "3":
        val = input("Masukkan limit speed (kosongkan untuk unlimited, cth 500K atau 2M): ").strip()
        GLOBAL_SETTINGS["rate_limit"] = val if val else None
    elif pilih == "4":
        val = input("Masukkan timeout (detik): ").strip()
        if val.isdigit():
            GLOBAL_SETTINGS["socket_timeout"] = val
    elif pilih == "5":
        GLOBAL_SETTINGS["playlist_mode"] = "no" if GLOBAL_SETTINGS["playlist_mode"] == "yes" else "yes"

def build_base_cmd():
    cmd = ["yt-dlp"]
    if CONFIG_FILE.exists():
        cmd.extend(["--config-locations", str(CONFIG_FILE)])

    cmd.extend(["-P", f"temp:{TEMP_DIR}"])
    cmd.extend(["-P", f"home:{DOWNLOADS_DIR}"])
    cmd.extend(["-o", "%(title)s [%(id)s].%(ext)s"])

    cmd.append("--no-keep-fragments")
    cmd.extend(["--parse-metadata", "%(uploader)s:%(meta_artist)s"])

    if GLOBAL_SETTINGS["use_aria2c"]:
        cmd.extend(["--downloader", "aria2c", "--downloader-args", f"aria2c:-x {GLOBAL_SETTINGS['concurrent_fragments']} -s {GLOBAL_SETTINGS['concurrent_fragments']}"])
    else:
        cmd.extend(["--concurrent-fragments", GLOBAL_SETTINGS["concurrent_fragments"]])

    if GLOBAL_SETTINGS["rate_limit"]:
        cmd.extend(["--limit-rate", GLOBAL_SETTINGS["rate_limit"]])

    cmd.extend(["--socket-timeout", GLOBAL_SETTINGS["socket_timeout"]])

    if GLOBAL_SETTINGS["playlist_mode"] == "no":
        cmd.append("--no-playlist")
    else:
        cmd.append("--yes-playlist")

    return cmd

def menu_resolusi_solid():
    print("\n" + "=" * 50)
    print("A. PILIH RESOLUSI UNDUHAN SOLID")
    print("=" * 50)
    print("AA. VIDEO + AUDIO")
    print("  1. Kualitas Terbaik Otomatis")
    print("  2. Kualitas 1440p / 2160p (4K) / 4320p (8K) Asli Video")
    print("  3. Kualitas 1080p Kunci Asli")
    print("  4. Kualitas 720p Kunci Asli")
    print("  5. Kualitas Rendah (140p / 144p / 240p / 360p / 480p)")
    print("BA. AUDIO SAJA")
    print("  6. Audio Terbaik (320kbps)")
    print("  7. High Audio")
    print("  8. Medium Audio")
    print("  9. Low Audio")
    print("CA. VIDEO SAJA (TANPA AUDIO)")
    print("  10. Video Saja Kualitas Terbaik")
    print("  11. Video Saja 1440p / 2160p / 4320p Asli")
    print("  12. Video Saja 1080p Kunci Asli")
    print("  13. Video Saja 720p Kunci Asli")
    print("  14. Video Saja Rendah (144p - 480p)")

    sub_pilihan = input("\nMasukkan Nomor Pilihan Resolusi (1-14): ").strip()

    fmt = "bestvideo+bestaudio/best"
    if sub_pilihan == "2":
        fmt = "bestvideo[height>=1440]+bestaudio/best"
    elif sub_pilihan == "3":
        fmt = "bestvideo[height<=1080][height>=1080]+bestaudio/best[height<=1080]"
    elif sub_pilihan == "4":
        fmt = "bestvideo[height<=720][height>=720]+bestaudio/best[height<=720]"
    elif sub_pilihan == "5":
        fmt = "bestvideo[height<=480]+bestaudio/best[height<=480]"
    elif sub_pilihan == "6":
        return ["-x", "--audio-format", "mp3", "--audio-quality", "0"]
    elif sub_pilihan == "7":
        return ["-x", "--audio-format", "m4a", "--audio-quality", "2"]
    elif sub_pilihan == "8":
        return ["-x", "--audio-format", "mp3", "--audio-quality", "5"]
    elif sub_pilihan == "9":
        return ["-x", "--audio-format", "mp3", "--audio-quality", "9"]
    elif sub_pilihan == "10":
        fmt = "bestvideo"
    elif sub_pilihan == "11":
        fmt = "bestvideo[height>=1440]"
    elif sub_pilihan == "12":
        fmt = "bestvideo[height<=1080][height>=1080]"
    elif sub_pilihan == "13":
        fmt = "bestvideo[height<=720][height>=720]"
    elif sub_pilihan == "14":
        fmt = "bestvideo[height<=480]"

    return ["-f", fmt]

def process_single_url(target_url, extra_args):
    cmd = build_base_cmd() + extra_args

    print("\n--- FITUR PROSESING TAMBAHAN ---")
    thumb = input("Sertakan Thumbnail Cover? (y/n, default=y): ").strip().lower()
    if thumb == 'n':
        cmd.append("--no-embed-thumbnail")
    else:
        cmd.append("--embed-thumbnail")

    print("\nPilih Konversi Container Video:")
    print("1. Default (MKV/MP4 Auto)  2. MP4  3. MKV  4. AVI  5. MOV  6. WEBM")
    c_opt = input("Pilihan Container (1-6): ").strip()
    container_map = {"2": "mp4", "3": "mkv", "4": "avi", "5": "mov", "6": "webm"}
    if c_opt in container_map:
        cmd.extend(["--merge-output-format", container_map[c_opt], "--recode-video", container_map[c_opt]])

    sub_opt = input("\nUnduh Takarir/Subtitle? (1. Input Langsung ke Video / 2. Unduh Terpisah / 3. Tanpa Subtitle): ").strip()
    if sub_opt == "1":
        lang = input("Masukkan kode bahasa (cth: id,en atau 'all'): ").strip() or "all"
        cmd.extend(["--embed-subs", "--sub-langs", lang])
    elif sub_opt == "2":
        lang = input("Masukkan kode bahasa (cth: id,en atau 'all'): ").strip() or "all"
        cmd.extend(["--write-subs", "--sub-langs", lang, "--skip-download"])

    clipper = input("\nIngin Memotong Segmen Video tertentu? (y/n): ").strip().lower()
    if clipper == 'y':
        start_t = input("Waktu Mulai (format HH:MM:SS atau Detik): ").strip()
        end_t = input("Waktu Selesai (format HH:MM:SS atau Detik): ").strip()
        cmd.extend(["--download-sections", f"*{start_t}-{end_t}"])

    live_opt = input("\nApakah Ini Video Siaran Langsung / Live Stream? (y/n): ").strip().lower()
    if live_opt == 'y':
        cmd.extend(["--from-title", ".*", "--live-from-start"])

    # Menggunakan Python API YoutubeDL dengan progress_hook jika tidak menggunakan aria2c external downloader
    if not GLOBAL_SETTINGS["use_aria2c"]:
        ydl_opts = {
            'progress_hooks': [progress_hook],
            'quiet': True,
            'noprogress': True,
            'paths': {'temp': str(TEMP_DIR), 'home': str(DOWNLOADS_DIR)},
            'outtmpl': '%(title)s [%(id)s].%(ext)s',
        }
        print(f"\n[*] Menjalankan pipeline unduhan untuk URL: {target_url}")
        try:
            with YoutubeDL(ydl_opts) as ydl:
                ydl.download([target_url])
            write_history(target_url, "BERHASIL", f"Format: {extra_args}")
            print(f"\n[+] Berhasil! File tersimpan di Google Drive: {DOWNLOADS_DIR}")
        except Exception as e:
            print(f"\n[!] Error eksekusi: {e}")
            write_history(target_url, "GAGAL", str(e))
    else:
        cmd.append(target_url)
        print(f"\n[*] Menjalankan pipeline unduhan untuk URL: {target_url}")
        try:
            subprocess.run(cmd, check=True)
            write_history(target_url, "BERHASIL", f"Format: {extra_args}")
            print(f"\n[+] Berhasil! File tersimpan di Google Drive: {DOWNLOADS_DIR}")
        except Exception as e:
            print(f"\n[!] Error eksekusi: {e}")
            write_history(target_url, "GAGAL", str(e))

def handle_atomic_logging():
    print("\n[*] Mengekstraksi Log Universal & Status Atomic Program...")
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    with open(VERBOSE_LOG, "w", encoding="utf-8") as f:
        f.write("=" * 80 + "\n")
        f.write(f"      SYSTEM DIAGNOSTIC & ATOMIC STATE LOG - {timestamp}\n")
        f.write("=" * 80 + "\n\n")

        f.write("[1] PYTHON & ENVIRONMENT PATH STATE\n")
        f.write(f"Python Executable : {sys.executable}\n")
        f.write(f"Python Version    : {sys.version}\n")
        f.write(f"Base Directory    : {DRIVE_BASE}\n")
        f.write(f"System PATH       : {os.environ.get('PATH')}\n\n")

        f.write("[2] BINARY DEPENDENCIES STATUS\n")
        for bin_name, desc in CORE_DEPENDENCIES.items():
            bin_path = shutil.which(bin_name)
            exists = bin_path is not None
            f.write(f"Binary: {bin_name:<20} | Status: {'OK' if exists else 'MISSING':<7} | Path: {bin_path}\n")
        f.write("\n")

        f.write("[3] RUNTIME EXECUTABLE VERSION DIAGNOSTICS\n")
        try:
            ytdlp_ver = subprocess.check_output(["yt-dlp", "--version"], text=True, stderr=subprocess.STDOUT).strip()
            f.write(f"yt-dlp Version    : {ytdlp_ver}\n")
        except Exception as e:
            f.write(f"yt-dlp Version    : ERROR ({e})\n")

        try:
            ffmpeg_ver = subprocess.check_output(["ffmpeg", "-version"], text=True, stderr=subprocess.STDOUT).splitlines()[0]
            f.write(f"FFmpeg Version    : {ffmpeg_ver}\n")
        except Exception as e:
            f.write(f"FFmpeg Version    : ERROR ({e})\n")
        f.write("\n")

        f.write("[4] CURRENT GLOBAL SETTINGS STATE\n")
        for k, v in GLOBAL_SETTINGS.items():
            f.write(f"Setting {k:<22} : {v}\n")
        f.write("\n")

        f.write("[5] CONFIGURATION FILE CONTENT (yt-dlp.conf)\n")
        if CONFIG_FILE.exists():
            f.write(CONFIG_FILE.read_text(encoding="utf-8") + "\n\n")
        else:
            f.write("FILE yt-dlp.conf TIDAK DITEMUKAN!\n\n")

        f.write("[6] RECENT DOWNLOAD HISTORY LOG\n")
        if HISTORY_LOG.exists():
            lines = HISTORY_LOG.read_text(encoding="utf-8").splitlines()
            f.write("\n".join(lines[-30:]) + "\n")
        else:
            f.write("Belum ada riwayat unduhan.\n")

        f.write("\n" + "=" * 80 + "\n")
        f.write("END OF DIAGNOSTIC LOG REPORT\n")
        f.write("=" * 80 + "\n")

    print(f"[+] Log status universal berhasil diekstrak penuh ke: {VERBOSE_LOG}")

def main():
    while True:
        print("\n" + "=" * 80)
        print("                  ULTIMATE VIDEO DOWNLOADER SUITE - MAIN MENU                  ")
        print("=" * 80)
        print("1. Unduh Single URL / Multi-Link Sekaligus (Proses Pipeline Utama)")
        print("2. Cek Verifikasi Dependencies Binary Linux")
        print("3. Manajemen Versi & Channel yt-dlp (Nightly/Stable/Master)")
        print("4. Pengaturan Mode Advanced (Aria2c, Limit Speed, Timeout, Fragmen, Playlist)")
        print("5. Ekstraksi File Log Detail Rinci (Atom Level Debug Logging)")
        print("6. Keluar dari Aplikasi")

        main_choice = input("\nPilih Menu Utama (1-6): ").strip()

        if main_choice == "2":
            check_binaries()
        elif main_choice == "3":
            manage_ytdlp_version()
        elif main_choice == "4":
            configure_advanced_options()
        elif main_choice == "5":
            handle_atomic_logging()
        elif main_choice == "6":
            print("\nTerima kasih telah menggunakan Ultimate Video Downloader Suite!")
            break
        elif main_choice == "1":
            print("\n--- INPUT TARGET URL ---")
            print("Anda bisa menempel satu URL atau banyak URL sekaligus (pisahkan dengan koma atau spasi).")
            raw_input_url = input("Masukkan URL Target: ").strip()
            if not raw_input_url:
                print("[!] URL tidak boleh kosong.")
                continue

            urls = [u.strip() for u in raw_input_url.replace(" ", ",").split(",") if u.strip()]

            print("\n--- METODE PEMILIHAN FORMAT ---")
            print("A. Pilih Resolusi Unduhan Solid (Daftar Preset Praktis)")
            print("B. Pilihan Resolusi Hasil Ekstrak Dinamis (-F Engine Inspector)")
            mode_fmt = input("Pilih Metode (A/B): ").strip().upper()

            extra_args = []
            if mode_fmt == "B":
                print("\n[*] Menjalankan Ekstraksi Engine yt-dlp -F terhadap URL pertama...\n")
                try:
                    res = subprocess.run(["yt-dlp", "-F", urls[0]], capture_output=True, text=True, check=True)
                    print(res.stdout)
                except subprocess.CalledProcessError as e:
                    print(f"[!] Gagal mengekstraksi format: {e}")
                    if e.stderr:
                        print(e.stderr)

                selected_id = input("\nMasukkan Format ID Pilihan Anda (contoh: 16+2 atau cukup masukkan 16): ").strip()
                if selected_id:
                    if selected_id.isdigit():
                        fmt_str = f"{selected_id}+bestaudio/best"
                    else:
                        fmt_str = selected_id
                else:
                    fmt_str = "bestvideo+bestaudio/best"
                extra_args = ["-f", fmt_str]
            else:
                extra_args = menu_resolusi_solid()

            print(f"\n[+] Total {len(urls)} URL terdeteksi.")
            exec_mode = "1"
            if len(urls) > 1:
                print("1. Proses Unduh Satu per Satu secara Berurutan")
                print("2. Proses Unduh Sekaligus (Batch URL Download)")
                exec_mode = input("Pilihan Eksekusi Batch (1/2): ").strip()

            if exec_mode == "2":
                for target in urls:
                    process_single_url(target, extra_args)
            else:
                for target in urls:
                    process_single_url(target, extra_args)

if __name__ == "__main__":
    main()

Mounted at /content/drive

                  ULTIMATE VIDEO DOWNLOADER SUITE - MAIN MENU                  
1. Unduh Single URL / Multi-Link Sekaligus (Proses Pipeline Utama)
2. Cek Verifikasi Dependencies Binary Linux
3. Manajemen Versi & Channel yt-dlp (Nightly/Stable/Master)
4. Pengaturan Mode Advanced (Aria2c, Limit Speed, Timeout, Fragmen, Playlist)
5. Ekstraksi File Log Detail Rinci (Atom Level Debug Logging)
6. Keluar dari Aplikasi

Pilih Menu Utama (1-6): 1

--- INPUT TARGET URL ---
Anda bisa menempel satu URL atau banyak URL sekaligus (pisahkan dengan koma atau spasi).
Masukkan URL Target: https://www.youtube.com/watch?v=JP12d7SiuMM

--- METODE PEMILIHAN FORMAT ---
A. Pilih Resolusi Unduhan Solid (Daftar Preset Praktis)
B. Pilihan Resolusi Hasil Ekstrak Dinamis (-F Engine Inspector)
Pilih Metode (A/B): B

[*] Menjalankan Ekstraksi Engine yt-dlp -F terhadap URL pertama...

[youtube] Extracting URL: https://www.youtube.com/watch?v=JP12d7SiuMM
[youtube] JP12d7SiuMM: Downloading we